# Huffman Canonical Implementation

In this Jupyter Notebook, the implementation of Canonical Huffman Coding is presented. The notebook includes the realization of the algorithm, a detailed explanation of its main steps, and an example which demonstrating the algorithm.

The purpose of this function is to read some .txt file and convert its characters into ascii values. 

In [2]:
def read_and_convert_file(file_name):
    numbers = []
    with open(file_name, "r", encoding="utf-8") as f:
        tekst = f.read()
        for char in tekst: 
            ascii_val = ord(char)
            if ascii_val < 256:
                numbers.append(ascii_val)
    return numbers

In this function we are sorting symbols in ascending order. 

In [3]:
import numpy as np

def sort_frequencies(numbers):
    freqs = [0] * 256
    syms = list(range(256))

    for input_symbol in numbers:
        match_idx = syms.index(input_symbol)
        new_freq = freqs[match_idx] + 1
        last_shift_idx = match_idx
        for i in range(match_idx, 256):
            if new_freq >= freqs[i]:
                last_shift_idx = i
            else:
                break

        for i in range(match_idx, last_shift_idx):
            freqs[i] = freqs[i + 1]
            syms[i] = syms[i + 1]

        freqs[last_shift_idx] = new_freq
        syms[last_shift_idx] = input_symbol

    sorted_active = []
    for f, s in zip(freqs, syms):
        if f > 0:
            sorted_active.append((f, s))

    return sorted_active

This function constructs the Huffman tree using two sorted lists: one containing the original symbol nodes and another containing the newly created internal nodes. The purpose of maintaining two separate lists is to avoid repeatedly sorting the list after each new node is created, since the internal nodes are generated in ascending order of their frequencies. The function returns the root node address together with the complete node table describing the tree structure.

In [4]:
def build_huffman_tree(active_nodes):
    tree_memory = {} # dictionary for creating the tree
    next_internal_id = 256  # id needs to be above 255 cause first 255 are reserved for symbols 
    
    symbol_list = active_nodes.copy() # sbram - our sorted symbol list
    internal_list = [] # ibram - in this list we are writing the new internal nodes who are always gonna be in sorted order
    
    def pop_min(): # this function finds the minimum in the both of the list
        if not symbol_list: # if one is empty, pop from the other one
            return internal_list.pop(0)
        if not internal_list:
            return symbol_list.pop(0)

        if symbol_list[0][0] <= internal_list[0][0]: # compare 
            return symbol_list.pop(0)
        else:
            return internal_list.pop(0)
            
    while len(symbol_list) + len(internal_list) > 1: # as long as the length above one the tree is not finished 
        freq1, id1 = pop_min() # first minimum
        freq2, id2 = pop_min() # second minimum
        
        new_freq = freq1 + freq2 # we are creating the new internal node
        new_id = next_internal_id
        next_internal_id += 1

        tree_memory[new_id] = {'left': id1, 'right': id2} # new_id acts as the parent id, storing its children
        
        internal_list.append((new_freq, new_id))
        
    root_addr = internal_list[0][1] # the only one who is left in the list represents the root of the whole tree
    
    print(f"2. The root is on the address: {root_addr}\n")
    return root_addr, tree_memory

This function traverses the Huffman tree from the root to all leaf nodes using a depth-first search (DFS) approach implemented with a stack. For each leaf node , it records the depth of the node, which corresponds to the length of its Huffman codeword, and returns a dictionary containing the code lengths for all symbols.

In [5]:
def tree_top_down_algorithm(root_addr, tree_memory):
    lengths_bram = {} # dictionary for symbols lengths
    stack = [(root_addr, 0)] # we are beginning from the root
    
    while stack: # until the stack is not empty, we did not finished 
        curr_addr, depth = stack.pop()
        
        if curr_addr < 256: # if the adress is under the 256, that means we come to leaf and thats our symbol(msb = 0)
            lengths_bram[curr_addr] = depth
        else: # if the adress is above the 255, we still propagate through the tree(msb = 1)
            children = tree_memory[curr_addr]
            stack.append((children['right'], depth + 1))
            stack.append((children['left'], depth + 1))
            
    return lengths_bram

This function creates a histogram of Huffman code lengths by counting the number of symbols assigned to each code length. This information is required for Canonical Huffman code generation, where the first codeword of each length is computed using the canonical coding formula, avoiding any additional traversal of the Huffman tree.

In [6]:
def count_length_frequencies(lengths_bram):
    max_length = max(lengths_bram.values()) # defining the maximum length 
    bl_count = {i: 0 for i in range(1, max_length + 1)} # creating dictionary to know how many symbols has the same length 
    
    for sym, length in lengths_bram.items():
        bl_count[length] += 1 # counting
        
    print("4. Length histogram :")
    for length, count in bl_count.items():
        if count > 0: 
            print(f" Length {length} -> {count} symbols ")
            
    return bl_count, max_length

This function generates the Canonical Huffman codes using the code-length histogram instead of traversing the Huffman tree. First, it calculates the starting codeword for each code length, then assigns consecutive binary values to symbols sorted by increasing code length and, for equal lengths, by their symbol value (ASCII order).
For each code length $L$, the initial binary code value is calculated based on the previous length $L-1$ using the following formula:

$$C_{L} = (C_{L-1} + N_{L-1}) \ll 1$$

Where:
* $C_{L}$ — The initial decimal code value for the current length $L$.
* $C_{L-1}$ — The initial code value for the previous length $L-1$.
* $N_{L-1}$ — The total number of symbols that have a code length of $L-1$ 

For the shortest possible code length , the value starts at zero:
$$C_{1} = 0$$

Once the initial code $C_{L}$ is determined for a specific length, each subsequent symbol of the same length is assigned a value incremented by $1$:
$$C_{L}^{(next)} = C_{L}^{(current)} + 1$$

In [7]:
def generate_canonical_codes(lengths_bram, bl_count, max_length): 
	next_code = {1: 0} 
	code = 0 

	for bits in range(1, max_length + 1): 
		code = (code + bl_count.get(bits - 1, 0)) << 1 
		next_code[bits] = code 

	dictionary_bram = {} 
	print("5. Dictionary BRAM :")
	for sym in range(256): 
		if sym in lengths_bram: 
			l = lengths_bram[sym] 
			assigned_code = next_code[l] 
			bin_code_str = format(assigned_code, f'0{l}b') 
			dictionary_bram[sym] = {'length': l, 'code_str': bin_code_str} 
			print(f" Symbol : ASCII {sym:<3} -> Binary code : {bin_code_str} (Length: {l})")
			next_code[l] += 1 
			
	return dictionary_bram


In [8]:
file_name = "input_text.txt"
symbols = read_and_convert_file(file_name)
active_nodes = sort_frequencies(symbols)
root_addr, tree_memory = build_huffman_tree(active_nodes)
lengths_bram = tree_top_down_algorithm(root_addr, tree_memory)
bl_count, max_length = count_length_frequencies(lengths_bram)
dictionary_bram = generate_canonical_codes(lengths_bram, bl_count, max_length)

2. The root is on the address: 300

4. Length histogram :
 Length 3 -> 2 symbols 
 Length 4 -> 7 symbols 
 Length 5 -> 5 symbols 
 Length 6 -> 7 symbols 
 Length 7 -> 3 symbols 
 Length 8 -> 1 symbols 
 Length 9 -> 6 symbols 
 Length 10 -> 4 symbols 
 Length 11 -> 5 symbols 
 Length 12 -> 6 symbols 
5. Dictionary BRAM :
 Symbol : ASCII 32  -> Binary code : 000 (Length: 3)
 Symbol : ASCII 39  -> Binary code : 1111111000 (Length: 10)
 Symbol : ASCII 44  -> Binary code : 110110 (Length: 6)
 Symbol : ASCII 45  -> Binary code : 11111111000 (Length: 11)
 Symbol : ASCII 46  -> Binary code : 1111010 (Length: 7)
 Symbol : ASCII 66  -> Binary code : 111111111010 (Length: 12)
 Symbol : ASCII 67  -> Binary code : 1111111001 (Length: 10)
 Symbol : ASCII 68  -> Binary code : 11111111001 (Length: 11)
 Symbol : ASCII 69  -> Binary code : 111110110 (Length: 9)
 Symbol : ASCII 70  -> Binary code : 111111111011 (Length: 12)
 Symbol : ASCII 75  -> Binary code : 11111111010 (Length: 11)
 Symbol : ASCII 77 

In [12]:
import numpy as np  # type: ignore
import serial  # type: ignore
import sys
import os

COM_PORT = 'COM10'
BAUD_RATE = 115200
FILE_NAME = "input_text.txt"
TOTAL_BYTES = 256 * 3 

def parse_uart_data(raw_data):
    uart_dict = {}
    for sym in range(256):
        idx = sym * 3
        length = raw_data[idx]
        
        if length > 0:
            code_high = raw_data[idx + 1]
            code_low = raw_data[idx + 2]
            code_val = (code_high << 8) | code_low
            code_str = f"{code_val:0{length}b}"
            
            uart_dict[sym] = {
                'code_str': code_str,
                'length': length
            }
    return uart_dict

def compare_dictionaries(sw_dict, hw_dict):
    print("\n" + "=" * 80)
    print(" COMPARE DICTIONARY SW VS HW")
    print("=" * 80)
    
    all_syms = sorted(list(set(sw_dict.keys()).union(set(hw_dict.keys()))))
    mismatches = 0
    
    for sym in all_syms:
        sw_data = sw_dict.get(sym, {'code_str': 'N/A', 'length': 0})
        hw_data = hw_dict.get(sym, {'code_str': 'N/A', 'length': 0})
        
        sw_code = sw_data['code_str']
        hw_code = hw_data['code_str']
        
        if sym == 32: char_repr = "'SPACE'"
        elif sym == 10: char_repr = "'\\n'"
        elif sym == 13: char_repr = "'\\r'"
        elif 32 < sym < 127: char_repr = f"'{chr(sym)}'"
        else: char_repr = f"ASCII_{sym}"

        match = (sw_code == hw_code) and (sw_data['length'] == hw_data['length'])
        status = "[ CORRECT ]" if match else "[ DIFFERENT! ]"
        if not match:
            mismatches += 1

        print(f"Symbol : {char_repr:<8} (ASCII {sym:<3d}) | SW: {sw_code:<14} (L:{sw_data['length']:2d}) | HW: {hw_code:<14} (L:{hw_data['length']:2d}) | {status}")
        
    print("=" * 80)
    if mismatches == 0:
        print("STATUS: THEY ARE SIMILAR")
    else:
        print(f"STATUS: FOUND {mismatches} DIFFERENCES")
    print("=" * 80 + "\n")
    return mismatches == 0

def compress_text(symbols, dictionary):
    bitstream = ""
    for sym in symbols:
        if sym not in dictionary:
            print(f"ERROR: Symbol ASCII {sym} does not exist in the dictionary!")
            return None
        bitstream += dictionary[sym]['code_str']
    return bitstream

def decompress_text(bitstream, dictionary):
    reverse_dict = {val['code_str']: sym for sym, val in dictionary.items()}
    
    decoded_symbols = []
    current_bits = ""
    
    for bit in bitstream:
        current_bits += bit
        if current_bits in reverse_dict:
            decoded_symbols.append(reverse_dict[current_bits])
            current_bits = ""
            
    if current_bits != "":
        print(f"WARNING: There were {len(current_bits)} uncompressed bits left at the end")

    return decoded_symbols


def main():
    print(f"Loading input text from '{FILE_NAME}'...")
    symbols = read_and_convert_file(FILE_NAME)
    print(f"Text successfully loaded ({len(symbols)} characters).")

    active_nodes = sort_frequencies(symbols)
    root_addr, tree_memory = build_huffman_tree(active_nodes)
    lengths_bram = tree_top_down_algorithm(root_addr, tree_memory)
    bl_count, max_length = count_length_frequencies(lengths_bram)
    dictionary_bram = generate_canonical_codes(lengths_bram, bl_count, max_length)

    print(f"\nOpening port {COM_PORT} at {BAUD_RATE} bps...")
    try:
        ser = serial.Serial(COM_PORT, BAUD_RATE, timeout=5)
        print("Waiting for data from FPGA (press the button on the board!)...")
        raw_data = ser.read(TOTAL_BYTES)
        ser.close()
    except Exception as e:
        print(f"Error opening COM port: {e}")
        return

    if len(raw_data) != TOTAL_BYTES:
        print(f"Entire dictionary was not received! Received {len(raw_data)} out of {TOTAL_BYTES} bytes.")
        return

    print(f" Successfully received {TOTAL_BYTES} bytes from FPGA hardware!")

    hw_dictionary = parse_uart_data(raw_data)
    compare_dictionaries(dictionary_bram, hw_dictionary)


    print("=" * 80)
    print(" COMPRESSION AND DECOMPRESSION TEST FOR ENTIRE INPUT TEXT")
    print("=" * 80)

    compressed_bitstream = compress_text(symbols, hw_dictionary)
    if compressed_bitstream is None:
        return

    decompressed_symbols = decompress_text(compressed_bitstream, hw_dictionary)

    is_exact_match = (symbols == decompressed_symbols)

    orig_str = "".join([chr(s) for s in symbols])
    decomp_str = "".join([chr(s) for s in decompressed_symbols])

    print(f"Original input     ({len(symbols)} characters) : {repr(orig_str[:60])}...")
    print(f"Decompressed output ({len(decompressed_symbols)} characters): {repr(decomp_str[:60])}...\n")

    if is_exact_match:
        print("SUCCESS: Entire input text and decompressed output are 100% IDENTICAL!")
    else:
        print(" ERROR: Decompressed text does NOT match the input text!")

    orig_bits = len(symbols) * 8
    comp_bits = len(compressed_bitstream)
    compression_ratio = orig_bits / comp_bits if comp_bits > 0 else 0
    space_savings = (1 - (comp_bits / orig_bits)) * 100 if orig_bits > 0 else 0

    print("\n" + "=" * 80)
    print(" COMPRESSION RATIO STATISTICS (FOR ENTIRE TEXT)")
    print("=" * 80)
    print(f" Input text size     : {orig_bits} bits ({len(symbols)} Bytes)")
    print(f" Compressed size     : {comp_bits} bits ({comp_bits / 8:.2f} Bytes)")
    print(f" Compression ratio (CR) : {compression_ratio:.2f}:1")
    print(f" Space savings       : {space_savings:.2f}%")
    print("=" * 80 + "\n")

if __name__ == '__main__':
    main()

Loading input text from 'input_text.txt'...
Text successfully loaded (5000 characters).
2. The root is on the address: 300

4. Length histogram :
 Length 3 -> 2 symbols 
 Length 4 -> 7 symbols 
 Length 5 -> 5 symbols 
 Length 6 -> 7 symbols 
 Length 7 -> 3 symbols 
 Length 8 -> 1 symbols 
 Length 9 -> 6 symbols 
 Length 10 -> 4 symbols 
 Length 11 -> 5 symbols 
 Length 12 -> 6 symbols 
5. Dictionary BRAM :
 Symbol : ASCII 32  -> Binary code : 000 (Length: 3)
 Symbol : ASCII 39  -> Binary code : 1111111000 (Length: 10)
 Symbol : ASCII 44  -> Binary code : 110110 (Length: 6)
 Symbol : ASCII 45  -> Binary code : 11111111000 (Length: 11)
 Symbol : ASCII 46  -> Binary code : 1111010 (Length: 7)
 Symbol : ASCII 66  -> Binary code : 111111111010 (Length: 12)
 Symbol : ASCII 67  -> Binary code : 1111111001 (Length: 10)
 Symbol : ASCII 68  -> Binary code : 11111111001 (Length: 11)
 Symbol : ASCII 69  -> Binary code : 111110110 (Length: 9)
 Symbol : ASCII 70  -> Binary code : 111111111011 (Lengt